# 🏠 Airbnb NYC 2019 – Sentiment Analysis
## Notebook 4: LLM — DistilBERT Zero-Shot & Fine-tuned Inference

**Pipeline:**
1. Use `distilbert-base-uncased-finetuned-sst-2-english` (Hugging Face) for binary sentiment
2. Compare with VADER labels
3. Confidence score distribution
4. Sample misclassification analysis
5. Optional: lightweight DistilBERT fine-tuning on the NYC dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix

BASE    = Path('.')
CLEANED = BASE / 'Dataset' / 'Cleaned Data'
PALETTE = {'POSITIVE': '#00e5ff', 'NEGATIVE': '#ff5252',
           'Positive': '#00e5ff', 'Neutral': '#ffd54f', 'Negative': '#ff5252'}
print('✅ Imports ready')

In [ ]:
df_full = pd.read_csv(CLEANED / 'reviews_cleaned.csv')
df_full.dropna(subset=['comments', 'sentiment'], inplace=True)

# Sample 500 reviews for LLM inference (cost/time sensitive)
df = df_full.sample(500, random_state=42).reset_index(drop=True)
print(f'Running inference on {len(df)} samples')
df['sentiment'].value_counts()

In [ ]:
# ── Load DistilBERT pipeline ─────────────────────────────────────────────────
print('Loading DistilBERT… (downloads on first run)')
sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    truncation=True,
    max_length=512
)
print('✅ Model loaded')

In [ ]:
# Run inference in batches
BATCH = 32
results = []
texts   = df['comments'].tolist()

for i in range(0, len(texts), BATCH):
    batch = texts[i:i+BATCH]
    preds = sentiment_pipeline(batch)
    results.extend(preds)
    if (i // BATCH) % 5 == 0:
        print(f'  Processed {min(i+BATCH, len(texts))}/{len(texts)}')

df['bert_label']      = [r['label']  for r in results]
df['bert_confidence'] = [r['score']  for r in results]

# Map DistilBERT binary → match VADER 3-class for Positive/Negative only
df['bert_mapped'] = df['bert_label'].map({'POSITIVE': 'Positive', 'NEGATIVE': 'Negative'})

df[['comments', 'sentiment', 'bert_label', 'bert_confidence']].head(10)

In [ ]:
# ── DistilBERT label distribution ────────────────────────────────────────────
bert_counts = df['bert_label'].value_counts().reset_index()
bert_counts.columns = ['Label', 'Count']

fig = px.pie(
    bert_counts, names='Label', values='Count',
    color='Label', color_discrete_map=PALETTE,
    title='<b>DistilBERT Sentiment Predictions</b>',
    template='plotly_dark', hole=0.45
)
fig.update_traces(textinfo='percent+label', textfont_size=14)
fig.update_layout(title_font_size=18)
fig.show()

In [ ]:
# ── Confidence Score Distribution ────────────────────────────────────────────
fig = px.histogram(
    df, x='bert_confidence', color='bert_label',
    color_discrete_map=PALETTE, nbins=40, marginal='violin',
    title='<b>DistilBERT Confidence Score Distribution</b>',
    template='plotly_dark'
)
fig.update_layout(title_font_size=18, xaxis_title='Confidence Score')
fig.show()

In [ ]:
# ── VADER vs DistilBERT comparison (excl. Neutral) ───────────────────────────
df_cmp = df[df['sentiment'] != 'Neutral'].copy()
df_cmp['agreement'] = df_cmp['sentiment'] == df_cmp['bert_mapped']

agree_pct = df_cmp['agreement'].mean() * 100
print(f'VADER ↔ DistilBERT agreement (excl. Neutral): {agree_pct:.1f}%')

# Crosstab heatmap
ct = pd.crosstab(df_cmp['sentiment'], df_cmp['bert_mapped'])

fig = px.imshow(
    ct.values, x=ct.columns.tolist(), y=ct.index.tolist(),
    text_auto=True, color_continuous_scale='Blues',
    title=f'<b>VADER vs DistilBERT Agreement  ({agree_pct:.1f}%)</b>',
    template='plotly_dark'
)
fig.update_layout(xaxis_title='DistilBERT', yaxis_title='VADER')
fig.show()

In [ ]:
# ── Misclassification Analysis ───────────────────────────────────────────────
mismatches = df_cmp[~df_cmp['agreement']].sort_values('bert_confidence', ascending=False)
print(f'Disagreements: {len(mismatches)}')

display_cols = ['comments', 'sentiment', 'bert_label', 'bert_confidence']
mismatches[display_cols].head(10)

In [ ]:
# ── High-confidence predictions showcase ────────────────────────────────────
high_conf = df[df['bert_confidence'] > 0.99].sort_values('bert_confidence', ascending=False)

fig = px.scatter(
    df,
    x=range(len(df)), y='bert_confidence',
    color='bert_label', color_discrete_map=PALETTE,
    hover_data=['comments'],
    title='<b>DistilBERT Confidence per Review</b>',
    template='plotly_dark', opacity=0.7
)
fig.update_layout(xaxis_title='Review Index', yaxis_title='Confidence',
                  title_font_size=18)
fig.show()

print('\n✅ Notebook 4 complete.')